In [0]:
import zipfile
import os
import shutil

# Correct Unity Catalog volume paths (use /Volumes/ not /dbfs/)
base_dir = '/Volumes/research_catalog/healthcare/policy_docs/cms_zip_docs'
output_dir = '/Volumes/research_catalog/healthcare/policy_docs/cms_all_docs'

os.makedirs(output_dir, exist_ok=True)

def extract_zip(zip_path, extract_to):
    """Extract a zip file to the target directory."""
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"  📦 Extracted: {os.path.basename(zip_path)}")

print("="*70)
print("🚀 Starting Recursive Extraction")
print("="*70)

# Extract the main zip file
print("\n1️⃣ Extracting main zip file...")
main_zip = os.path.join(base_dir, 'all_data.zip')
extract_zip(main_zip, base_dir)

# Recursively extract ALL zip files in subfolders
print("\n2️⃣ Searching for nested zip files...")
zip_count = 0
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith('.zip') and file != 'all_data.zip':  # Skip the main zip
            zip_path = os.path.join(root, file)
            extract_zip(zip_path, root)
            zip_count += 1

print(f"✅ Extracted {zip_count} nested zip files")

# Collect ALL PDFs and CSVs from all subdirectories
print("\n3️⃣ Collecting PDFs and CSVs from all subdirectories...")
pdf_count = 0
csv_count = 0

for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check for PDF or CSV files
        if file.lower().endswith('.pdf'):
            src = os.path.join(root, file)
            dst = os.path.join(output_dir, file)
            # Handle duplicate filenames
            if os.path.exists(dst):
                base, ext = os.path.splitext(file)
                counter = 1
                while os.path.exists(dst):
                    dst = os.path.join(output_dir, f"{base}_{counter}{ext}")
                    counter += 1
            shutil.move(src, dst)
            pdf_count += 1
        
        elif file.lower().endswith('.csv'):
            src = os.path.join(root, file)
            dst = os.path.join(output_dir, file)
            # Handle duplicate filenames
            if os.path.exists(dst):
                base, ext = os.path.splitext(file)
                counter = 1
                while os.path.exists(dst):
                    dst = os.path.join(output_dir, f"{base}_{counter}{ext}")
                    counter += 1
            shutil.move(src, dst)
            csv_count += 1

print("\n" + "="*70)
print("✨ EXTRACTION COMPLETE")
print("="*70)
print(f"📄 PDFs collected:  {pdf_count}")
print(f"📊 CSVs collected:  {csv_count}")
print(f"📁 Output location: {output_dir}")
print("="*70)
print("\n💡 Next: Use CSVs for structured relationships (Document_IDs)")
print("💡      Use PDFs for unstructured content (clinical details)")
print("="*70)

In [0]:
import os
import shutil

# Source: extracted files
source_dir = '/Volumes/research_catalog/healthcare/policy_docs/cms_all_docs'
# Destination: GraphRAG input folder
target_dir = '/Volumes/research_catalog/healthcare/policy_docs/input'

# Create target directory if it doesn't exist
os.makedirs(target_dir, exist_ok=True)

print("="*70)
print("📦 Moving Files to GraphRAG Input Folder")
print("="*70)
print(f"Source:      {source_dir}")
print(f"Destination: {target_dir}")
print("="*70)

# Get all files from source
source_files = os.listdir(source_dir)

moved_count = 0
skipped_count = 0
error_count = 0

for filename in source_files:
    source_path = os.path.join(source_dir, filename)
    target_path = os.path.join(target_dir, filename)
    
    # Skip if target already exists
    if os.path.exists(target_path):
        print(f"⏭️  Skipped (exists): {filename}")
        skipped_count += 1
        continue
    
    try:
        shutil.move(source_path, target_path)
        file_type = "📊 CSV" if filename.endswith('.csv') else "📄 PDF"
        print(f"✅ Moved {file_type}: {filename}")
        moved_count += 1
    except Exception as e:
        print(f"❌ Error moving {filename}: {str(e)}")
        error_count += 1

print("\n" + "="*70)
print("✨ MOVE COMPLETE")
print("="*70)
print(f"✅ Moved:   {moved_count} files")
print(f"⏭️  Skipped: {skipped_count} files (already exist)")
print(f"❌ Errors:  {error_count} files")
print(f"📁 Location: {target_dir}")
print("="*70)
print("\n💡 Ready to process with GraphRAG in notebook 2197913251262124")
print("="*70)